In [ ]:
## importing packages
from enum import auto
from statistics import median
import statistics
from unicodedata import decimal
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from pprint import pprint
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
from hyperopt import hp, fmin, tpe, space_eval, Trials
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
import os
from sklearn.neural_network import MLPRegressor
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense
import random as python_random


In [ ]:
# Seed value
# Apparently you may use different seed values at each stage
def reset_seeds():
   np.random.seed(9) 
   python_random.seed(9)
   tf.random.set_seed(9)


In [ ]:
## Open the file
Perovskites = pd.read_csv(r"C:\Users\Perovskites - machine learning\database_perovskitas_ML_multitarget_hotencoding_importantvariabNOVO.txt", 
                                delimiter = "\t", decimal = ",")

## remove columns based on correlation plot, 'perovskite' não dá pra classificar só em dois tipos
##  e 'am/ca' (ver caderno o pq)
Perovskites = Perovskites.drop(["Total_amount"], axis=1)

## remove rows outliers and stuff
Perovskites = Perovskites.drop([37,52,154, 56, 15, 65], axis=0, inplace=False)

#### columns removed based on shap feature importance
Perovskites = Perovskites.drop(['S_II','S_II_amount_mL'], axis=1)

### usando one hot encoding em algumas categóricas
Perovskites_encoded = pd.get_dummies(Perovskites, columns=['A_source','B1_source', 'B2_source', 'X_source'])
print(Perovskites_encoded)

#trocando features para logfeatures - features com range muito alto
Perovskites_encoded['Diameter_nm'] = np.log10(Perovskites_encoded["Diameter_nm"])
Perovskites_encoded['Time_min'] = np.log10(Perovskites_encoded["Time_min"])
Perovskites_encoded['B2_amount_mmol'] = np.log10(Perovskites_encoded["B2_amount_mmol"])
Perovskites_encoded['Temperature_K'] = np.log10(Perovskites_encoded["Temperature_K"])
Perovskites_encoded['Bandgap'] = np.log10(Perovskites_encoded["Bandgap"])
Perovskites_encoded['Tolerance_factor_new'] = np.log10(Perovskites_encoded["Tolerance_factor_new"])

## after remove columns from shap
x = Perovskites_encoded.drop(['B1_amount_mmol', 'X_amount_mmol', 'Time_min', 'Bandgap', 'Amine_amount_mmol'], axis=1)
y = Perovskites_encoded[[ 'B1_amount_mmol',  'X_amount_mmol', 'Time_min','Bandgap', 'Amine_amount_mmol']]

np.random.seed(100)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=25) # 75% training and 25% test

In [ ]:
'''
### optimization using hyperopt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error # Importar a função mean_squared_error
# Data preprocessing
scaler = MinMaxScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)


from hyperopt import fmin, tpe, hp, Trials

# Define a função objetivo que recebe os hiperparâmetros e retorna a métrica de desempenho
def objective(params):
  # Desempacote os hiperparâmetros
  units1 = params['units1']
  units2 = params['units2']
  units3 = params['units3']
  epochs = params['epochs']

  # Crie e compile o modelo com os hiperparâmetros
  reset_seeds()
  model = Sequential()
  model.add(Dense(units1, input_dim=76, kernel_initializer='he_uniform', activation='relu'))
  model.add(Dense(units2, kernel_initializer='he_uniform', activation='relu'))
  model.add(Dense(units3, kernel_initializer='he_uniform', activation='relu'))
  model.add(Dense(5))
  model.compile(loss ='mean_squared_error', optimizer = 'adam') # Substituir a função de perda por mean_squared_error

  # Treine o modelo e obtenha a métrica de desempenho no conjunto de teste
  history = model.fit(X_train_norm, y_train, verbose = 'auto', epochs = epochs, shuffle=False, use_multiprocessing=False)
  y_pred = model.predict(X_test_norm)
  rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Substituir a métrica de desempenho por mean_squared_error e tirar a raiz quadrada

  # Retorne a métrica de desempenho
  return rmse

# Defina o espaço de busca dos hiperparâmetros
space = {
  'units1': hp.choice('units1', [100, 200]),
  'units2': hp.choice('units2', [100,150]),
  'units3': hp.choice('units3', [50,80]),
  'epochs': hp.choice('epochs', [100, 150])
}

# Crie um objeto Trials para armazenar os resultados
trials = Trials()

# Use o algoritmo TPE para encontrar os melhores hiperparâmetros que minimizam a função objetivo
best = fmin(fn=objective,
            space=space,
            algo=tpe.suggest,
            max_evals=50,
            trials=trials)

# Imprima os melhores hiperparâmetros
print(best)

from hyperopt import space_eval
best_values = space_eval(space, best)
print(best_values)
'''

In [ ]:
#for hyperopt
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.fit_transform(X_test)

def reset_seeds():
   np.random.seed(9) 
   python_random.seed(9)
   tf.random.set_seed(9)

reset_seeds() 

model = Sequential()
model.add(Dense((200), input_dim=76, kernel_initializer='he_uniform', activation='relu'))
model.add(Dense(150, kernel_initializer='he_uniform', activation='relu'))
model.add(Dense(80, kernel_initializer='he_uniform', activation='relu'))
model.add(Dense(5))

model.compile(loss ='mae', optimizer = 'adam')
model.summary()
history = model.fit(X_train_norm, y_train, verbose = 'auto',epochs = 100, shuffle=False, use_multiprocessing=False)

# prediction on test set
y_pred=model.predict(X_test_norm)

## training metrics
y_pred_train=model.predict(X_train_norm)

In [ ]:
'''
## hyperparameter optimization using GridSearch
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import GridSearchCV

## hyperparameter optimization using GridSearch
def create_model(hidden_layer1=200, hidden_layer2=150, hidden_layer3=80):
    reset_seeds()
    model = Sequential()
    model.add(Dense(hidden_layer1, input_dim=76, kernel_initializer='he_uniform', activation='relu'))
    model.add(Dense(hidden_layer2, kernel_initializer='he_uniform', activation='relu'))
    model.add(Dense(hidden_layer3, kernel_initializer='he_uniform', activation='relu'))
    model.add(Dense(5))
    model.compile(loss ='mae', optimizer='adam')
    return model

model = KerasRegressor(build_fn=create_model, verbose=0)

param_grid = {
    'hidden_layer1': range(100, 200, 20),
    'hidden_layer2': [50, 150, 30],
    'hidden_layer3': [30, 80, 20],
    'epochs': [50, 200, 50]
}

scaler = MinMaxScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='neg_mean_absolute_error', cv=5)
grid_result = grid.fit(X_train_norm, y_train)

print("Melhores hiperparâmetros: ", grid_result.best_params_)
print("Melhor resultado de MAE: ", -grid_result.best_score_)
'''


In [ ]:
'''
### for grid search

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.fit_transform(X_test)

def reset_seeds():
   np.random.seed(9) 
   python_random.seed(9)
   tf.random.set_seed(9)

reset_seeds() 

model = Sequential()
model.add(Dense((100), input_dim=76, kernel_initializer='he_uniform', activation='relu'))
model.add(Dense(30, kernel_initializer='he_uniform', activation='relu'))
model.add(Dense(30, kernel_initializer='he_uniform', activation='relu'))
model.add(Dense(5))

model.compile(loss ='mae', optimizer = 'adam')
model.summary()
history = model.fit(X_train_norm, y_train, verbose = 'auto',epochs = 200, shuffle=False, use_multiprocessing=False)

# prediction on test set
y_pred=model.predict(X_test_norm)

## training metrics
y_pred_train=model.predict(X_train_norm)
'''

In [ ]:
MAE_train = pd.DataFrame(metrics.mean_absolute_error(y_train, y_pred_train, multioutput='raw_values'))
RMSE_train = pd.DataFrame(np.sqrt(metrics.mean_squared_error(y_train, y_pred_train, multioutput='raw_values')))
R2_train = pd.DataFrame(r2_score(y_train, y_pred_train, multioutput='raw_values'))

train_metrics_nn = pd.concat([MAE_train,  RMSE_train, R2_train], axis='columns')
train_metrics_nn.columns = ['MAE_train', 'RMSE_train','R2_train']
print(train_metrics_nn)

### test metrics
MAE_test= pd.DataFrame(metrics.mean_absolute_error(y_test, y_pred, multioutput='raw_values'))
RMSE_test = pd.DataFrame(np.sqrt(metrics.mean_squared_error(y_test, y_pred, multioutput='raw_values')))
R2_test =pd.DataFrame(r2_score(y_test, y_pred, multioutput='raw_values'))

test_metrics_nn = pd.concat([MAE_test, RMSE_test, R2_test], axis='columns')
test_metrics_nn.columns = ['MAE_test','RMSE_test','R2_test']
print(test_metrics_nn)